<img height="100" src="https://i.postimg.cc/gjptBxF4/logo-gas-removebg-preview.png" width="250"/>

| Country        | States                                                       | Producing Regions                                                                        | Productivity Data | Soil File | Average Cycle |
|----------------|--------------------------------------------------------------|------------------------------------------------------------------------------------------|-------------------|-----------|---------------|
| United States  | Iowa, Illinois, Nebraska, Minnesota, Indiana                 | Corn Belt (IA, IL, IN); East/Center of NE; South of MN                                   | USDA              | EC6       | Apr – Nov     |
| China          | Heilongjiang, Jilin, Nei Mongol, Shandong, Henan             | Northeast and North China Plains                                                         | NBS               | EC6       | Apr – Oct     |
| Brazil         | Mato Grosso, Paraná, Goiás, Mato Grosso do Sul, Minas Gerais | MT (Mid-North), PR (West), GO (South), MS (Southwest), MG (Triangle)                     | SIDRA-IBGE        | EC3       | Jan – Sep     |
| European Union | France, Romania, Poland, Hungary, Italy                      | FRA (N. Aquitaine), ROM (South), POL (Lower Silesia), HUN (Great Plain), ITA (Po Valley) | AGRI4CAST         | EC2       | Mar – Dec     |
| Argentina      | Córdoba, Buenos Aires, Santa Fé, Santiago del Estero         | Core Zone (North BA, South SF, Center CD); Southeast SDE                                 | BC EXPLORER       | EC6       | Sep – Aug     |
| India          | Karnataka, Madhya Pradesh, Bihar, Tamil Nadu, Telangana      | Ballari-KA, Chhindwara-MP, "Corn Zone"-BI                                                | DES               | EC4       | Mar – Dec     |
| Mexico         | Sinaloa, Jalisco, Michoacán, Guerrero, Chiapas               | Sinaloa Valleys; Ciénega/Altos Regions (Jalisco)                                         | DGSIAP            | EC4       | Apr – Feb     |

Soil File Legend:

* EC2: medium texture soils
* EC3: medium-fine soils
* EC4: fine soils
* EC6: fine and permeable soils <br>
The average cycle comprises the period from sowing to harvest.

In [ ]:
import os

import pandas as pd

path = os.path.join(os.getcwd(),"inputs",'data','coordinates.xlsx')
df_address = pd.read_excel(path)

# Brasil - BR (SIDRA-IBGE)
https://apisidra.ibge.gov.br/ - Tabela 5457

In [ ]:
import time
import sidrapy

# Defining Brazil addresses
df_BR = df_address.query("country == 'Brazil'")
df = pd.read_excel(path, sheet_name='SIDRA-ids')

# 1. Recria o dataframe a partir dos dados fornecidos pelo usuário
df_locs = df.copy()

# Remove duplicatas de códigos de município para não buscar o mesmo ID várias vezes
municipios_unicos = df_locs[['cod_municipio', 'name']].drop_duplicates()

# 2. Itera sobre cada município para buscar os data
all_yield_data = []
total_municipios = len(municipios_unicos)

print(f"Iniciando busca da série histórica de rendimento de milho para {total_municipios} municípios...")

for index, row in municipios_unicos.iterrows():
    cod_municipio = str(row['cod_municipio'])
    nome_municipio = row['name']

    print(f"({index+1}/{total_municipios}) Buscando data para: {nome_municipio} ({cod_municipio})...", end="")

    try:
        data = sidrapy.get_table(
            table_code="5457",
            territorial_level="6",
            ibge_territorial_code=cod_municipio,
            variable="112",
            classifications={"782": "40122"},
            period="all"
        )

        if data is not None and len(data) > 1:
            # Pula o cabeçalho e adiciona o código do município para referência
            df_municipio = data.iloc[1:].copy()
            df_municipio['cod_municipio'] = int(cod_municipio)
            all_yield_data.append(df_municipio)
            print(" Sucesso.")
        else:
            print(" Sem data retornados.")

    except Exception as e:
        print(f" Falha. Erro: {e}")

    # Pausa para não sobrecarregar a API
    time.sleep(0.5)

# 3. Consolida e limpa os data
if all_yield_data:
    print("\nConsolidando todos os data...")
    df_final = pd.concat(all_yield_data, ignore_index=True)

    # Renomeia as colunas para nomes mais claros
    df_final = df_final.rename(columns={
        'D1C': 'cod_municipio_api', # Código do Município (retornado pela API)
        'D1N': 'municipio_nome',    # Nome do Município
        'D2N': 'ano',               # Ano
        'V': 'yield (kg/ha)'        # Valor do rendimento
    })

    # Seleciona apenas as colunas de interesse
    df_final = df_final[['cod_municipio', 'municipio_nome', 'ano', 'yield (kg/ha)']]

    # Converte tipos de data e remove linhas com valores nulos
    df_final['ano'] = pd.to_numeric(df_final['ano'], errors='coerce')
    df_final['yield (kg/ha)'] = pd.to_numeric(df_final['yield (kg/ha)'], errors='coerce')
    df_final.dropna(inplace=True)

    # Converte para inteiros
    df_final['country'] = 'Brazil'
    df_final['ano'] = df_final['ano'].astype(int)
    df_final['yield (kg/ha)'] = df_final['yield (kg/ha)'].astype(int)

    # 4. Salva o resultado em um arquivo Excel
    xlsx_path = path
    print(f"Salvando data consolidados em '{xlsx_path}'...")

    try:
        # Tenta ler o arquivo existente para adicionar a nova aba sem apagar as outras
        with pd.ExcelFile(xlsx_path) as xls:
            sheets = {sheet_name: pd.read_excel(xls, sheet_name) for sheet_name in xls.sheet_names}
    except FileNotFoundError:
        sheets = {} # Se o arquivo não existe, cria um dicionário vazio

    # Adiciona/substitui a aba com os novos data
    sheets['yield_BR'] = df_final

    with pd.ExcelWriter(xlsx_path, engine='openpyxl') as writer:
        for sheet_name, df_sheet in sheets.items():
            df_sheet.to_excel(writer, sheet_name=sheet_name, index=False)

    print("\nOperação finalizada com sucesso!")
    print("Prévia dos data:")
    print(df_final.head())

else:
    print("\nNenhum dado foi obtido na busca.")

In [ ]:
import pandas as pd

print("A preparar a junção dos dados do Brasil...")

# ============================================================
# 1. PADRONIZAR A CHAVE DE LIGAÇÃO
# ============================================================
# Garantir que o cod_municipio seja lido como string para um match perfeito
df_locs['cod_municipio'] = df_locs['cod_municipio'].astype(str)
df_final['cod_municipio'] = df_final['cod_municipio'].astype(str)

# ============================================================
# 2. FAZER O MERGE (JUNÇÃO)
# ============================================================
# Unimos df_locs (lat, lon, state, etc) com df_final (histórico de yield)
df_brasil_consolidado = pd.merge(
    df_locs[['lat', 'lon', 'country', 'state', 'cod_municipio', 'name']], # Dados da Esquerda (Localização)
    df_final[['cod_municipio', 'ano', 'yield (kg/ha)']],                  # Dados da Direita (Produtividade)
    on='cod_municipio',
    how='inner'
)

# ============================================================
# 3. LIMPAR E FORMATAR A TABELA FINAL
# ============================================================
# Renomear 'name' para 'county' (ou municipio) para manter o padrão das colunas
df_brasil_consolidado.rename(columns={
    'name': 'county'
}, inplace=True)

# Reordenar para o padrão: lat, lon, country, state, county/municipio, ano, yield
df_brasil_consolidado = df_brasil_consolidado[['lat', 'lon', 'country', 'state', 'county', 'ano', 'yield (kg/ha)']]

# ============================================================
# 4. SALVAR O RESULTADO
# ============================================================
print("\nJunção concluída com sucesso! Eis uma amostra:")
print(df_brasil_consolidado.head())

# Salvar no Excel na aba final consolidada do Brasil
with pd.ExcelWriter(path, mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
    df_brasil_consolidado.to_excel(writer, sheet_name='yield_BR', index=False)

print("\nTabela final do Brasil guardada na aba 'DADOS_MILHO_BR_FINAL'.")

# United States - USA (USDA)

In [ ]:
df_USA = df_address.query("country=='United States'")
df_USA.head().reset_index(drop=True)

In [ ]:
import requests
import os
import time
import pandas as pd

# ============================================================
# 1. CONFIGURAÇÕES
# ============================================================
# COLOCA A TUA CHAVE AQUI (Pede em: https://quickstats.nass.usda.gov/api)
USDA_API_KEY = '8ECF0CEB-1783-39BF-9375-F500ABBD6B94'

# ============================================================
# 2. GEOCODING REVERSO (Pontos -> Condados FIPS)
# ============================================================
def mapear_condados_eua(caminho_excel):
    print("A carregar os pontos do Excel...")
    df = pd.read_excel(caminho_excel, sheet_name='Sheet1') # Ajusta o nome da aba se necessário

    # Filtrar apenas pontos dos Estados Unidos
    df_usa = df[df['country'].str.contains('United States', na=False)].copy()

    if df_usa.empty:
        print("Nenhum ponto dos EUA encontrado.")
        return None

    gdf_pontos = gpd.GeoDataFrame(
        df_usa,
        geometry=gpd.points_from_xy(df_usa.lon, df_usa.lat),
        crs="EPSG:4326"
    )

    print("A carregar mapa de condados dos EUA (FIPS)...")
    # Este GeoJSON público contém as fronteiras de todos os condados dos EUA com o respetivo código FIPS (id)
    url_counties = "https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json"

    # O Geopandas consegue ler GeoJSON diretamente da web
    gdf_counties = gpd.read_file(url_counties)

    print("A associar pontos aos condados (Spatial Join)...")
    # Cruza os teus pontos com o mapa dos condados
    pontos_com_condado = gpd.sjoin(gdf_pontos, gdf_counties, how="inner", predicate='intersects')

    # Limpar e renomear para ficar organizado
    pontos_com_condado.rename(columns={'id': 'fips_code', 'NAME': 'county_name'}, inplace=True)

    return pontos_com_condado

# ============================================================
# 3. EXTRAÇÃO DE DADOS DO USDA NASS
# ============================================================
def buscar_yield_usda(estado_nome):
    """
    Busca a série histórica de rendimento de milho (em Bushels/Acre)
    para todos os condados de um determinado estado americano.
    """
    print(f"A consultar USDA NASS para o Estado: {estado_nome}...")

    url = "http://quickstats.nass.usda.gov/api/api_GET/"

    parametros = {
        'key': USDA_API_KEY,
        'source_desc': 'SURVEY',
        'sector_desc': 'CROPS',
        'commodity_desc': 'CORN',
        'utilization_practice_desc': 'GRAIN', # Apenas milho grão (exclui silagem)
        'statisticcat_desc': 'YIELD',
        'agg_level_desc': 'COUNTY',
        'state_name': estado_nome.upper(),    # A API exige maiúsculas (ex: IOWA)
        'format': 'JSON'
    }

    resposta = requests.get(url, params=parametros)

    if resposta.status_code == 200:
        dados = resposta.json()['data']
        df_usda = pd.DataFrame(dados)

        # O FIPS é a junção do código do Estado (state_ansi) + Condado (county_ansi)
        df_usda['fips_code'] = df_usda['state_ansi'] + df_usda['county_ansi']

        # Limpar colunas e converter rendimento
        df_limpo = df_usda[['fips_code', 'county_name', 'year', 'Value']].copy()
        df_limpo.rename(columns={'Value': 'yield_bu_acre', 'year': 'ano'}, inplace=True)

        # O USDA retorna valores numéricos como texto com vírgulas. Limpar isso:
        df_limpo['yield_bu_acre'] = pd.to_numeric(df_limpo['yield_bu_acre'].str.replace(',', ''), errors='coerce')
        df_limpo.dropna(subset=['yield_bu_acre'], inplace=True)
        df_limpo['ano'] = df_limpo['ano'].astype(int)

        return df_limpo
    else:
        print(f"Erro na API do USDA para {estado_nome}: {resposta.status_code}")
        return None

# ============================================================
# 4. EXECUÇÃO PRINCIPAL
# ============================================================
if __name__ == "__main__":
    caminho_ficheiro = os.path.join(os.getcwd(), 'inputs', 'data', 'coordinates.xlsx')

    try:
        # 1. Encontrar o código FIPS (Condado) para cada coordenada
        df_pontos_fips = mapear_condados_eua(caminho_ficheiro)

        if df_pontos_fips is not None:
            # Quais são os estados americanos que temos nos nossos dados? (ex: Iowa, Illinois)
            estados_unicos = df_pontos_fips['state'].unique()

            todos_dados_usda = []

            # 2. Buscar os dados na API para cada Estado
            for estado in estados_unicos:
                df_yield_estado = buscar_yield_usda(estado)
                if df_yield_estado is not None:
                    todos_dados_usda.append(df_yield_estado)
                time.sleep(1) # Pausa amigável para a API

            # 3. Consolidar tudo
            if todos_dados_usda:
                df_usda_final = pd.concat(todos_dados_usda, ignore_index=True)

                # 4. Converter Bushels/Acre para Kg/Ha (1 bushel de milho = ~25.4 kg; 1 acre = ~0.404 ha)
                # Logo: (Bu/Ac) * 62.77 = Kg/Ha
                df_usda_final['yield_kg_ha'] = (df_usda_final['yield_bu_acre'] * 62.77).round(0).astype(int)
                df_usda_final['country'] = 'United States'

                # Filtrar apenas as colunas de interesse
                df_usda_final = df_usda_final[['fips_code', 'county_name', 'ano', 'yield_kg_ha', 'country']]

                print("\nDados obtidos com sucesso! Guardando...")

                # Salvar na nova aba
                with pd.ExcelWriter(caminho_ficheiro, mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
                    # Salva o mapa de pontos + FIPS
                    df_pontos_fips[['lat', 'lon', 'country', 'state', 'county_name', 'fips_code']].to_excel(writer, sheet_name='FIPS_USA', index=False)
                    # Salva a série histórica
                    df_usda_final.to_excel(writer, sheet_name='yield_USA', index=False)

                print("Operação finalizada! Abas 'FIPS_USA' e 'yield_USA' criadas.")
            else:
                print("Não foi possível extrair dados de produtividade.")

    except Exception as e:
        print(f"Ocorreu um erro: {e}")

In [ ]:
# ============================================================
# 1. CARREGAR OS DADOS (Se estiverem no Excel)
# ============================================================
# caminho_ficheiro = "inputs/data/coordinates.xlsx"
df_pontos = pd.read_excel(caminho_ficheiro, sheet_name='FIPS_USA')
df_yield = pd.read_excel(caminho_ficheiro, sheet_name='yield_USA')

# Assumindo que já tens os DataFrames na memória como df_pontos_fips e df_usda_final:
df_pontos = df_pontos_fips.copy()
df_yield = df_usda_final.copy()

print("A preparar a junção dos dados...")

# ============================================================
# 2. PADRONIZAR A CHAVE DE LIGAÇÃO
# ============================================================
# É crucial garantir que o fips_code é lido como texto (string) em ambas as tabelas
# para evitar que um "19021" não dê match com um 19021 numérico.
df_pontos['fips_code'] = df_pontos['fips_code'].astype(str)
df_yield['fips_code'] = df_yield['fips_code'].astype(str)

# ============================================================
# 3. FAZER O MERGE (JUNÇÃO)
# ============================================================
# Vamos unir as colunas de interesse da tabela de pontos com a tabela de produtividade
# how='inner' garante que só mantemos os pontos que têm dados de produtividade associados.
df_final = pd.merge(
    df_pontos[['lat', 'lon', 'country', 'state', 'fips_code']], # Dados da Esquerda
    df_yield[['fips_code', 'county_name', 'ano', 'yield_kg_ha']], # Dados da Direita
    on='fips_code',
    how='inner'
)

# ============================================================
# 4. LIMPAR E FORMATAR A TABELA FINAL
# ============================================================
# Renomear as colunas para o padrão exato que pediste
df_final.rename(columns={
    'county_name': 'county',
    'yield_kg_ha': 'yield (kg/ha)'
}, inplace=True)

# Reordenar as colunas para a apresentação final
df_final = df_final[['lat', 'lon', 'country', 'state', 'county', 'ano', 'yield (kg/ha)']]

# ============================================================
# 5. SALVAR O RESULTADO
# ============================================================
print("\nJunção concluída com sucesso! Eis uma amostra:")
print(df_final.head())

# Salvar numa aba final para o modelo de Machine Learning / ERA5
with pd.ExcelWriter(caminho_ficheiro, mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
    df_final.to_excel(writer, sheet_name='yield_USA', index=False)

print("\nTabela final guardada na aba 'DADOS_MILHO_EUA_FINAL'.")

# China - CH (NBS)
https://data.stats.gov.cn/english/adv.htm?cn=C01

In [ ]:
import os
import pandas as pd
import geopandas as gpd

# ============================================================
# 1. CONFIGURAÇÕES DOS CAMINHOS
# ============================================================
caminho_excel = os.path.join('inputs', 'data', 'coordinates.xlsx')

# Caminho para o Shapefile descarregado do GADM (Nível 3 - Condados/Municípios)
# Ajusta o nome da pasta conforme o que descarregaste
caminho_shapefile_china = os.path.join('inputs', 'data','shape','gadm41_CHN_3.shp')

print("A iniciar o mapeamento de municípios para a China...")

def mapear_municipios_china(caminho_dados, caminho_shp):
    try:
        print("A ler os pontos do Excel...")
        df_address = pd.read_excel(caminho_dados, sheet_name='Sheet1')
        df_china = df_address[df_address['country'] == 'China'].copy()

        if df_china.empty:
            print("Nenhum ponto da China encontrado.")
            return None

        # Converter a tua malha para formato espacial
        gdf_pontos = gpd.GeoDataFrame(
            df_china,
            geometry=gpd.points_from_xy(df_china.lon, df_china.lat),
            crs="EPSG:4326"
        )

        # ==========================================================
        # A CORREÇÃO: Apagar a coluna 'state' antiga e mentirosa
        # ==========================================================
        if 'state' in gdf_pontos.columns:
            gdf_pontos = gdf_pontos.drop(columns=['state'])

        print("A carregar as fronteiras dos municípios chineses (GADM)...")
        gdf_municipios = gpd.read_file(caminho_shp)

        print("A cruzar a malha 0.25x0.25 com as fronteiras reais...")
        pontos_mapeados = gpd.sjoin(gdf_pontos, gdf_municipios, how="inner", predicate='intersects')

        # ==========================================================
        # EXTRAIR A VERDADE DO MAPA
        # NAME_1 = Província (State)
        # NAME_3 = Município (County)
        # ==========================================================
        pontos_mapeados.rename(columns={'NAME_1': 'state', 'NAME_3': 'county'}, inplace=True)

        # Filtrar apenas as colunas estruturais limpas
        df_final_locs = pontos_mapeados[['lat', 'lon', 'country', 'state', 'county']]

        return df_final_locs

    except Exception as e:
        print(f"Ocorreu um erro: {e}")
        return None

# ============================================================
# 2. EXECUÇÃO
# ============================================================
if __name__ == "__main__":
    # 1. Faz o mapeamento espacial (Spatial Join)
    df_locs_china = mapear_municipios_china(caminho_excel, caminho_shapefile_china)

    if df_locs_china is not None:
        # 2. Definir as províncias do escopo do projeto (Northeast and North China Plains)
        provincias_alvo = ['Heilongjiang', 'Jilin', 'Nei Mongol', 'Shandong', 'Henan']

        # 3. Aplicar o filtro
        print(f"\nA aplicar filtro de escopo para as províncias alvo...")
        df_locs_china_filtrado = df_locs_china[df_locs_china['state'].isin(provincias_alvo)].copy()

        # 4. Relatório do filtro
        print("\nMapeamento e filtragem concluídos com sucesso! Eis uma amostra:")
        print(df_locs_china_filtrado.head())
        print(f"\nTotal de pontos na China (sem filtro): {len(df_locs_china)}")
        print(f"Total de pontos no escopo do projeto: {len(df_locs_china_filtrado)}")

        # Nota: O dataframe `df_locs_china_filtrado` está agora pronto na memória.
        # NENHUM DADO FOI GUARDADO NO EXCEL AINDA.

        # O próximo passo será carregar a tabela de produtividade (yield) da China,
        # fazer o merge com este df_locs_china_filtrado e, SÓ ENTÃO, gravar a aba final!

In [ ]:
import pandas as pd
import os

print("A restaurar os dados da China a partir do ficheiro bruto...")

# ============================================================
# 1. LER O FICHEIRO ORIGINAL BRUTO
# ============================================================
# Usamos o caminho exato onde o teu ficheiro original está guardado
caminho_bruto = r'D:\OneDrive\Documentos\Git\Doutorado\inputs\data\yield\yield_CH.xls'

# Lê o Excel (assumindo que a linha 1 tem os anos e a palavra 'Region')
df_raw = pd.read_excel(caminho_bruto, skiprows=3)

# ============================================================
# 2. TRANSFORMAR E LIMPAR (Melt)
# ============================================================
# Transforma as colunas de anos em linhas
df_melted = df_raw.melt(
    id_vars=['Region'],
    var_name='ano',
    value_name='yield (kg/ha)'
)

# Limpar linhas nulas (caso o Excel tenha linhas em branco no final)
df_melted = df_melted.dropna(subset=['Region', 'yield (kg/ha)'])

# Converter a coluna ano para número inteiro para não dar problemas
df_melted['ano'] = pd.to_numeric(df_melted['ano'], errors='coerce')
df_melted = df_melted.dropna(subset=['ano']) # Remove cabeçalhos extra se existirem
df_melted['ano'] = df_melted['ano'].astype(int)

# ============================================================
# 3. O RESGATE DE NEI MONGOL (TRADUÇÃO)
# ============================================================
correcoes_nomes = {
    'Inner Mongolia': 'Nei Mongol'
}
df_melted['Region'] = df_melted['Region'].replace(correcoes_nomes)

# ============================================================
# 4. FILTRAR O ESCOPO DO PROJETO
# ============================================================
provincias_alvo = ['Heilongjiang', 'Jilin', 'Nei Mongol', 'Shandong', 'Henan']
df_historico_puro = df_melted[df_melted['Region'].isin(provincias_alvo)].copy()

print("\nProvíncias recuperadas com sucesso e prontas para o merge:")
print(df_historico_puro['Region'].unique())

# ============================================================
# 5. O MERGE FINAL
# ============================================================
print("\nA cruzar com as tuas coordenadas geográficas...")
# Assumindo que o df_locs_china_filtrado ainda está na tua memória do Python
df_china_final = pd.merge(
    df_locs_china_filtrado[['lat', 'lon', 'country', 'state', 'county']],
    df_historico_puro,
    left_on='state',
    right_on='Region',
    how='inner'
)

# ============================================================
# 6. FORMATAR E SALVAR
# ============================================================
df_china_final = df_china_final[['lat', 'lon', 'country', 'state', 'county', 'ano', 'yield (kg/ha)']]

# Salvar definitivamente na aba correta do teu ficheiro de coordenadas
caminho_excel = os.path.join(os.getcwd(), 'inputs', 'data', 'coordinates.xlsx')
with pd.ExcelWriter(caminho_excel, mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
    df_china_final.to_excel(writer, sheet_name='yield_CH', index=False)

print("\nChina finalizada com 100% de sucesso! 🎉")

# European Union - UE (AGRI4CAST)
https://agri4cast.jrc.ec.europa.eu/dataportal

In [ ]:
import os
import pandas as pd
import geopandas as gpd

data_path = os.path.join(os.getcwd(), 'inputs', 'data')
caminho_excel = os.path.join(data_path, 'coordinates.xlsx')

print("A iniciar o processamento da Europa (Com Hierarquia State/County Real)...")

# ============================================================
# 1. CARREGAR DICIONÁRIO DE NOMES NUTS (O SEGREDO DA HIERARQUIA)
# ============================================================
NUTS_path = os.path.join(data_path, 'yield', 'NUTS.xlsx')
df_NUTS = pd.read_excel(NUTS_path)

# Criamos um dicionário em memória: {'FRI1': 'Aquitaine', 'PL51': 'Dolnośląskie', ... }
nuts_nomes_dict = dict(zip(df_NUTS['NUTS Code'].astype(str), df_NUTS['NUTS label'].astype(str)))

# ============================================================
# 2. CARREGAR E FILTRAR COORDENADAS
# ============================================================
df_address = pd.read_excel(caminho_excel, sheet_name='Sheet1')
paises_ue = ['France', 'Poland', 'Italy', 'Romania', 'Hungary']
df_ue = df_address[df_address['country'].isin(paises_ue)].reset_index(drop=True).copy()

gdf_pontos_ue = gpd.GeoDataFrame(
    df_ue,
    geometry=gpd.points_from_xy(df_ue.lon, df_ue.lat),
    crs="EPSG:4326"
).reset_index(drop=True)

# ============================================================
# 3. GEOCODING COM MAPA NUTS 2016
# ============================================================
print("A cruzar com o mapa NUTS 2016...")
url_nuts_2016 = "https://gisco-services.ec.europa.eu/distribution/v2/nuts/geojson/NUTS_RG_01M_2016_4326_LEVL_3.geojson"
gdf_nuts = gpd.read_file(url_nuts_2016)

pontos_mapeados_ue = gpd.sjoin(gdf_pontos_ue, gdf_nuts, how="inner", predicate='intersects')

df_locs_ue = pontos_mapeados_ue[['lat', 'lon', 'country', 'NUTS_ID', 'NAME_LATN']].copy()
df_locs_ue.rename(columns={'NUTS_ID': 'IDREGION_MAPA', 'NAME_LATN': 'county'}, inplace=True)

# ============================================================
# EXTRAÇÃO INTELIGENTE DO STATE (NUTS 2 / NUTS 1)
# ============================================================
# County = NUTS 3 (ex: Landes)
# State = Procuramos o nome do NUTS 2 (4 letras) ou NUTS 1 (3 letras) no teu Excel NUTS
df_locs_ue['state'] = df_locs_ue['IDREGION_MAPA'].str[:4].map(nuts_nomes_dict)
# Caso não encontre no nível 2, tenta o nível 1 (3 letras) como fallback
df_locs_ue['state'] = df_locs_ue['state'].fillna(df_locs_ue['IDREGION_MAPA'].str[:3].map(nuts_nomes_dict))
# Se ainda assim falhar, coloca o nome do País
df_locs_ue['state'] = df_locs_ue['state'].fillna(df_locs_ue['country'])

# ============================================================
# 4. LER O AGRI4CAST
# ============================================================
print("A carregar os dados do AGRI4CAST...")
agri4cast_path = os.path.join(data_path, 'yield', 'agri4cast_2_ef3b6d63-f4d0-4860-a0f2-5c65cdeaf0a6_2025.v01_39.xlsx')
df_yield = pd.read_excel(agri4cast_path)
df_yield['yield (kg/ha)'] = pd.to_numeric(df_yield['yield (kg/ha)'], errors='coerce').fillna(0).astype(int)

# Filtro de Escopo do projeto
prefixos_alvo = ('FRI', 'RO31', 'RO41', 'PL51', 'HU3', 'ITC1', 'ITC4', 'ITH3', 'ITH5')
df_yield_filtrado = df_yield[df_yield['IDREGION'].astype(str).str.startswith(prefixos_alvo)].copy()

# ============================================================
# 5. MATCHER HIERÁRQUICO COGNITIVO
# ============================================================
codigos_agri4cast = set(df_yield_filtrado['IDREGION'].unique())

def encontrar_melhor_nuts(nuts_mapa):
    if nuts_mapa in codigos_agri4cast:
        return nuts_mapa
    elif nuts_mapa[:4] in codigos_agri4cast:
        return nuts_mapa[:4]
    elif nuts_mapa[:3] in codigos_agri4cast:
        return nuts_mapa[:3]
    return None

df_locs_ue['IDREGION_JOIN'] = df_locs_ue['IDREGION_MAPA'].apply(encontrar_melhor_nuts)
df_locs_ue = df_locs_ue.dropna(subset=['IDREGION_JOIN'])

# ============================================================
# 6. O MERGE FINAL
# ============================================================
print("A executar a fusão final...")
df_eu_final = pd.merge(
    df_locs_ue,
    df_yield_filtrado[['IDREGION', 'YEAR', 'yield (kg/ha)']],
    left_on='IDREGION_JOIN',
    right_on='IDREGION',
    how='inner'
)

# Padronização das 7 colunas sagradas (Garantindo a coluna 'ano')
df_eu_final.rename(columns={'YEAR': 'ano'}, inplace=True)
df_eu_final = df_eu_final[['lat', 'lon', 'country', 'state', 'county', 'ano', 'yield (kg/ha)']]

# ============================================================
# 7. RESUMO E GRAVAÇÃO
# ============================================================
print("\n=== RESUMO DA BASE EUROPEIA CORRIGIDA ===")
print(f"Total de registos criados: {len(df_eu_final)}")
print("\nAmostra do DataFrame resultante (Hierarquia Real):")
print(df_eu_final.head())

print("\nContagem por País:")
print(df_eu_final['country'].value_counts())

with pd.ExcelWriter(caminho_excel, mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
    df_eu_final.to_excel(writer, sheet_name='yield_EU', index=False)

print("\nBase da Europa salva com sucesso na aba 'yield_EU'!")

# India - IN
http://data.icrisat.org/dld/src/crops.html

In [ ]:
import pandas as pd
import geopandas as gpd
import os

print("A iniciar o mapeamento da Índia com a base de dados ICRISAT/DES...")

data_path = os.path.join(os.getcwd(), 'inputs', 'data')
caminho_excel = os.path.join(data_path, 'coordinates.xlsx')
caminho_shp_in = os.path.join(data_path, 'shape', 'gadm41_IND_2.shp')
yield_path = os.path.join(data_path, 'yield', 'yield_IN.csv')

# ============================================================
# 1. CARREGAR E LIMPAR A NOVA BASE DE PRODUTIVIDADE
# ============================================================
df_yield = pd.read_csv(yield_path, sep=';',decimal='.')

# Renomear colunas para facilitar o uso no nosso padrão
df_yield = df_yield.rename(columns={
    'State Name': 'State',
    'Dist Name': 'District',
    'Year': 'ano',
    'MAIZE YIELD (Kg per ha)': 'yield (kg/ha)'
})

# Garantir formato numérico
df_yield['ano'] = pd.to_numeric(df_yield['ano'], errors='coerce')
df_yield['yield (kg/ha)'] = pd.to_numeric(df_yield['yield (kg/ha)'], errors='coerce')
df_yield = df_yield.dropna(subset=['ano', 'yield (kg/ha)'])
df_yield['ano'] = df_yield['ano'].astype(int)

# Padronizar nomes (o "Teste do Algodão": tudo maiúsculas e sem espaços extra)
df_yield['State_clean'] = df_yield['State'].astype(str).str.strip().str.upper()
df_yield['District_clean'] = df_yield['District'].astype(str).str.strip().str.upper()

# ============================================================
# 2. CARREGAR COORDENADAS DA TUA MALHA
# ============================================================
df_address = pd.read_excel(caminho_excel, sheet_name='Sheet1')
df_in = df_address[df_address['country'] == 'India'].copy()

gdf_pontos_in = gpd.GeoDataFrame(
    df_in,
    geometry=gpd.points_from_xy(df_in.lon, df_in.lat),
    crs="EPSG:4326"
)

# ============================================================
# 3. O MAPA OFICIAL E O FILTRO DE ESCOPO
# ============================================================
print("A cruzar coordenadas com o mapa GADM (Distritos)...")
gdf_mapa_in = gpd.read_file(caminho_shp_in)

# Padronizar o mapa da mesma forma que a base de dados
gdf_mapa_in['State_clean'] = gdf_mapa_in['NAME_1'].astype(str).str.strip().str.upper()
gdf_mapa_in['District_clean'] = gdf_mapa_in['NAME_2'].astype(str).str.strip().str.upper()

pontos_mapeados_in = gpd.sjoin(gdf_pontos_in, gdf_mapa_in, how="inner", predicate='intersects')

# Aplicar o filtro estrito do teu projeto (nos 5 estados alvo)
estados_alvo = ['KARNATAKA', 'MADHYA PRADESH', 'BIHAR', 'TAMIL NADU', 'TELANGANA']
pontos_mapeados_in = pontos_mapeados_in[pontos_mapeados_in['State_clean'].isin(estados_alvo)]

df_locs_in = pontos_mapeados_in[['lat', 'lon', 'country', 'NAME_1', 'NAME_2', 'State_clean', 'District_clean']].copy()
df_locs_in.rename(columns={'NAME_1': 'state', 'NAME_2': 'county'}, inplace=True)

# ============================================================
# 4. O GRANDE MERGE FINAL
# ============================================================
print("A fundir o histórico de produtividade (Yield)...")
df_in_final = pd.merge(
    df_locs_in,
    df_yield[['State_clean', 'District_clean', 'ano', 'yield (kg/ha)']],
    on=['State_clean', 'District_clean'],
    how='inner'
)

# Limpeza para as 7 colunas sagradas do teu Doutoramento
df_in_final = df_in_final[['lat', 'lon', 'country', 'state', 'county', 'ano', 'yield (kg/ha)']]

# ============================================================
# 5. RESUMO E GRAVAÇÃO
# ============================================================
print(f"\n=== RESUMO DA BASE DA ÍNDIA ===")
print(f"Total de registos criados: {len(df_in_final)}")
print("\nEstados presentes na base final:")
print(df_in_final['state'].value_counts())

with pd.ExcelWriter(caminho_excel, mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
    df_in_final.to_excel(writer, sheet_name='yield_IN', index=False)

print("\nÍndia finalizada com 100% de sucesso na aba 'yield_IN'! 🎉")

# Argentina - AR

https://www.bolsadecereales.com/ <br>
https://www.bolsadecereales.com/imagenes/pass/2025-09/1106-pas202509045.pdf

In [ ]:
import pandas as pd
import os

yield_path = os.path.join(os.getcwd(),'inputs','data','yield','yield_AR.csv')
df_ar = pd.read_csv(yield_path, encoding='latin1', sep=';')
caminho_excel = os.path.join('inputs','data', 'coordinates.xlsx')
df = pd.read_excel(caminho_excel, sheet_name='Sheet1')

In [ ]:
df_ar.head(20)

In [ ]:
df.loc[df.state=='Buenos Aires'].reset_index(drop=True).head(20)

In [ ]:
import pandas as pd
import geopandas as gpd
import os

print("A aplicar o Método Híbrido: Geografia Municipal + Produtividade Regional...")

data_path = os.path.join(os.getcwd(), 'inputs', 'data')
yield_path = os.path.join(data_path, 'yield', 'yield_AR.csv')
caminho_excel = os.path.join('inputs','data', 'coordinates.xlsx')
caminho_shp_arg = os.path.join(data_path, 'shape', 'gadm41_ARG_2.shp')

# ------------------------------------------------------------
# FUNÇÃO DEFENSIVA: Salva "Córdoba" e resolve problemas de acentos
# ------------------------------------------------------------
def remover_acentos_e_padronizar(coluna):
    return coluna.astype(str).str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8').str.strip().str.upper()

# ============================================================
# 1. PREPARAR HISTÓRICO REGIONAL (MAGyP)
# ============================================================
df_ar = pd.read_csv(yield_path, encoding='latin1', sep=';')
df_ar = df_ar[df_ar['Cultivo'] == 'Maíz'].copy()

def extrair_ano_colheita(campanha_str):
    partes = str(campanha_str).split('/')
    prefixo = partes[0][:2]
    sufixo = partes[1]
    ano_final = sufixo if len(sufixo) == 4 else prefixo + sufixo
    return int(ano_final)

df_ar['ano'] = df_ar['Campaña'].apply(extrair_ano_colheita)
df_ar.rename(columns={'Rendimiento': 'yield (kg/ha)'}, inplace=True)

# Aplicar a limpeza anti-acentos na província
df_ar['state_clean'] = remover_acentos_e_padronizar(df_ar['Provincia'])

# ============================================================
# 2. GEOCODING: OBTER MUNICÍPIOS EXATOS VIA SHAPEFILE
# ============================================================
df_coords = pd.read_excel(caminho_excel, sheet_name='Sheet1')
df_ar_coords = df_coords[df_coords['country'] == 'Argentina'].copy()

gdf_pontos_ar = gpd.GeoDataFrame(
    df_ar_coords,
    geometry=gpd.points_from_xy(df_ar_coords.lon, df_ar_coords.lat),
    crs="EPSG:4326"
)

print("A ler o mapa GADM para extrair os Municípios (Departamentos)...")
gdf_mapa_arg = gpd.read_file(caminho_shp_arg)

# Cruzamento Espacial (Spatial Join)
pontos_mapeados_ar = gpd.sjoin(gdf_pontos_ar, gdf_mapa_arg, how="inner", predicate='intersects')

# Agora temos a Geografia perfeita! (NAME_1 = Província, NAME_2 = Município)
df_locs_ar = pontos_mapeados_ar[['lat', 'lon', 'country', 'NAME_1', 'NAME_2']].copy()
df_locs_ar.rename(columns={'NAME_1': 'state', 'NAME_2': 'county'}, inplace=True)

# Limpar o nome do estado vindo do mapa para cruzar com o CSV
df_locs_ar['state_clean'] = remover_acentos_e_padronizar(df_locs_ar['state'])

# ============================================================
# 3. O MERGE HÍBRIDO
# ============================================================
print("A fundir a geografia detalhada com a produtividade regional...")
df_argentina_final = pd.merge(
    df_locs_ar,
    df_ar[['state_clean', 'ano', 'yield (kg/ha)']],
    on='state_clean',
    how='inner'
)

# Isolar as tuas 7 colunas sagradas
df_argentina_final = df_argentina_final[['lat', 'lon', 'country', 'state', 'county', 'ano', 'yield (kg/ha)']]

# ============================================================
# 4. RESUMO E GRAVAÇÃO
# ============================================================
print("\n=== RESUMO DA BASE DA ARGENTINA ===")
print(f"Total de registos criados: {len(df_argentina_final)}")
print("\nEstados que cruzaram com sucesso:")
print(df_argentina_final['state'].value_counts())

with pd.ExcelWriter(caminho_excel, mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
    df_argentina_final.to_excel(writer, sheet_name='yield_AR', index=False)

print("\nFeito! Aba 'yield_AR' atualizada com Municípios corretos e Córdoba resgatada! 🇦🇷")

# Mexico - MX
https://nube.agricultura.gob.mx/datosAbiertos/Agricola.php

In [ ]:
import pandas as pd
import os

years = [i for i in range(2003, 2025, 1)]
prefix = 'Cierre_agricola_mun_'
yield_path = os.path.join(os.getcwd(),'inputs','data','yield')

MX_yields_path = []
for year in years:
    MX_yields_path.append(os.path.join(yield_path, f'{prefix}{year}.csv'))

# --- Correção para eliminar DtypeWarning ---
dtype_spec = {
    'Rendimiento': 'str',
    'Volumenproduccion': 'str'
}

df_MX = pd.concat([
    pd.read_csv(
        path,
        encoding='ISO-8859-1',
        low_memory=False,
        dtype=dtype_spec
    )
    for path in MX_yields_path
], ignore_index=True)

# Converter para numérico após ler tudo
df_MX['Rendimiento'] = pd.to_numeric(df_MX['Rendimiento'], errors='coerce')
if 'Volumenproduccion' in df_MX.columns:
    df_MX['Volumenproduccion'] = pd.to_numeric(df_MX['Volumenproduccion'], errors='coerce')

# ---- Seu filtro continua igual ----
mask = {
    'Nomestado': ['Sinaloa', 'Jalisco', 'Michoacán', 'Guerrero', 'Chiapas'],
    'Nomcultivo': 'Maíz grano',
    'Nomcicloproductivo': 'Primavera-Verano',
    'Nommodalidad':'Temporal'
}

filtro = (
    df_MX['Nomestado'].isin(mask['Nomestado']) &
    (df_MX['Nomcultivo'] == mask['Nomcultivo']) &
    (df_MX['Nomcicloproductivo'] == mask['Nomcicloproductivo']) &
    (df_MX['Nommodalidad'] == mask['Nommodalidad'])
)

df_MX_filtrado = df_MX[filtro]

df_MX_final = df_MX_filtrado[['Anio','Nomestado','Nommunicipio','Nommodalidad','Rendimiento']]
df_MX_final.rename(columns=dict(
    Anio='ano',
    Nomestado='estado',
    Nommunicipio='municipio',
    Nommodalidad='modalidade',
    Rendimiento='yield (kg/ha)'
), inplace=True)

In [ ]:
df_MX_final['yield (kg/ha)'] = df_MX_final['yield (kg/ha)'].apply(lambda x: x * 1000)

df_MX_final.drop(columns='modalidade', inplace=True)

In [ ]:
df_MX_final = (
    df_MX_final
    .sort_values(by=['estado', 'municipio', 'ano'])
    .reset_index(drop=True)
)

In [ ]:
import os

# Cria o caminho correto
output_path = os.path.join(os.getcwd(),'inputs','data','yield','yield_MX.csv')

# Salva o arquivo CSV
df_MX_final.to_csv(output_path, index=False, encoding='utf-8')

In [ ]:
import pandas as pd
import geopandas as gpd
import os

print("A executar o Cruzamento Final do México...")

data_path = os.path.join(os.getcwd(), 'inputs', 'data')
caminho_excel = os.path.join('inputs','data', 'coordinates.xlsx')
caminho_shp_mx = os.path.join(data_path, 'shape', 'gadm41_MEX_2.shp')

# ============================================================
# 1. CARREGAR A TUA BASE LIMPA DO MÉXICO
# ============================================================
# Assumindo que guardaste este teu dataframe exato como yield_MX.csv
yield_path = os.path.join(os.getcwd(),'inputs','data','yield','yield_MX.csv')
df_mx = pd.read_csv(yield_path)

# Renomear para o padrão global (state e county)
df_mx.rename(columns={
    'estado': 'state',
    'municipio': 'county'
}, inplace=True)

# Converter yield para inteiro por segurança
df_mx['yield (kg/ha)'] = pd.to_numeric(df_mx['yield (kg/ha)'], errors='coerce').fillna(0).astype(int)

# Função Anti-Acentos para cruzamento seguro
def remover_acentos(coluna):
    return coluna.astype(str).str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8').str.strip().str.upper()

df_mx['state_clean'] = remover_acentos(df_mx['state'])
df_mx['county_clean'] = remover_acentos(df_mx['county'])

# ============================================================
# 2. GEOCODING COM MAPA DO MÉXICO (Nível 2)
# ============================================================
print("A ler o mapa oficial e as tuas coordenadas...")
df_coords = pd.read_excel(caminho_excel, sheet_name='Sheet1')
df_mx_coords = df_coords[df_coords['country'] == 'Mexico'].copy()

gdf_pontos_mx = gpd.GeoDataFrame(
    df_mx_coords,
    geometry=gpd.points_from_xy(df_mx_coords.lon, df_mx_coords.lat),
    crs="EPSG:4326"
)

gdf_mapa_mx = gpd.read_file(caminho_shp_mx)

# Padronizar nomes do mapa
gdf_mapa_mx['state_clean'] = remover_acentos(gdf_mapa_mx['NAME_1'])
gdf_mapa_mx['county_clean'] = remover_acentos(gdf_mapa_mx['NAME_2'])

# O Spatial Join
pontos_mapeados_mx = gpd.sjoin(gdf_pontos_mx, gdf_mapa_mx, how="inner", predicate='intersects')

df_locs_mx = pontos_mapeados_mx[['lat', 'lon', 'country', 'NAME_1', 'NAME_2', 'state_clean', 'county_clean']].copy()
df_locs_mx.rename(columns={'NAME_1': 'state', 'NAME_2': 'county'}, inplace=True)

# ============================================================
# 3. O MERGE FINAL
# ============================================================
print("A fundir as coordenadas com a produtividade...")
df_mexico_final = pd.merge(
    df_locs_mx,
    df_mx[['state_clean', 'county_clean', 'ano', 'yield (kg/ha)']],
    on=['state_clean', 'county_clean'],
    how='inner'
)

# Manter apenas as 7 colunas do Doutoramento (com os nomes geográficos limpos vindo do mapa)
df_mexico_final = df_mexico_final[['lat', 'lon', 'country', 'state', 'county', 'ano', 'yield (kg/ha)']]

# ============================================================
# 4. RESUMO E GRAVAÇÃO
# ============================================================
print("\n=== RESUMO DA BASE DO MÉXICO ===")
print(f"Total de registos criados: {len(df_mexico_final)}")
print("\nTop Municípios mapeados:")
print(df_mexico_final['county'].value_counts().head())

#Excluindo linhas com yield igual a zero (caso existam)
df_mexico_final = df_mexico_final[df_mexico_final['yield (kg/ha)'] > 0].reset_index(drop=True)

with pd.ExcelWriter(caminho_excel, mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
    df_mexico_final.to_excel(writer, sheet_name='yield_MX', index=False)

print("\nMéxico finalizado! A tua base global está completa. 🇲🇽")

In [ ]:
df_mexico_final.loc[df_mexico_final['county'] == 'Huetamo']

In [ ]:
import numpy as np

# 1. Substitui eventuais zeros falhos por NaN para não puxar a média para baixo
df_mexico_final['yield (kg/ha)'] = df_mexico_final['yield (kg/ha)'].replace(0, np.nan)

# 2. Agrupa por célula geográfica e tira a média (Transforma 28 linhas em 8 perfeitas)
df_mexico_final = df_mexico_final.groupby(
    ['lat', 'lon', 'country', 'state', 'county', 'ano'],
    as_index=False
)['yield (kg/ha)'].mean()

# 3. Restaura possíveis NaNs para 0 e converte para número inteiro
df_mexico_final['yield (kg/ha)'] = df_mexico_final['yield (kg/ha)'].fillna(0).round(0).astype(int)

df_mexico_final

In [ ]:
import plotly.express as px

# É recomendável filtrar por um ano específico (ex: 2021) para o mapa não sobrepor os anos
df_mapa = df_mexico_final[df_mexico_final['ano'] == 2021].copy()

# Cria o mapa interativo
fig = px.scatter_map(
    df_mapa,
    lat="lat",
    lon="lon",
    hover_name="county",
    hover_data=["state", "yield (kg/ha)"],
    color="yield (kg/ha)",
    size_max=15,
    zoom=4.5,
    center={"lat": 23.6345, "lon": -102.5528}, # Centro aproximado do México
    map_style="carto-positron",
    title="Distribuição da Produtividade na Grade ERA5 (0.25°) - México"
)

fig.update_layout(margin={"r":0,"t":40,"l":0,"b":0})
fig.show()

# Associating Yield and Coordinates

In [ ]:
import os

data_path = os.path.join(os.getcwd(),'inputs','data','coordinates.xlsx')

In [ ]:
import pandas as pd

df_BR = pd.read_excel(data_path, sheet_name='yield_BR')
df_USA = pd.read_excel(data_path, sheet_name='yield_USA')
df_CH = pd.read_excel(data_path, sheet_name='yield_CH')
df_UE = pd.read_excel(data_path, sheet_name='yield_EU')
df_IN = pd.read_excel(data_path, sheet_name='yield_IN')
df_AR = pd.read_excel(data_path, sheet_name='yield_AR')
df_MX = pd.read_excel(data_path, sheet_name='yield_MX')

In [ ]:
import pandas as pd
import numpy as np

print("=== INICIALIZANDO CONCATENAÇÃO E HARMONIZAÇÃO TEMPORAL ===")

# 1. Empacotar os DataFrames que já estão na memória
lista_dfs = [df_BR, df_USA, df_CH, df_UE, df_IN, df_AR, df_MX]

# 2. Concatenação Direta (O "empilhamento" bruto)
print("Empilhando os dados de todos os países...")
df_bruto = pd.concat(lista_dfs, ignore_index=True)

# Substituir 0 reais por NaN temporariamente para não distorcer as médias espaciais
df_bruto['yield (kg/ha)'] = df_bruto['yield (kg/ha)'].replace(0, np.nan)

# 3. Tratamento Espacial Definitivo (O Seguro de Vida)
print("Aplicando agrupamento espacial para consolidar micro-registros...")
df_consolidado = df_bruto.groupby(
    ['lat', 'lon', 'country', 'state', 'county', 'ano'],
    as_index=False
)['yield (kg/ha)'].mean()

# ==============================================================================
# 4. A MÁGICA DA HARMONIZAÇÃO TEMPORAL (BALANCED PANEL)
# ==============================================================================
print("Expandindo a linha do tempo (Harmonização Temporal)...")

# Descobre o ano mais antigo e o mais novo do MUNDO
ano_min = 1940
ano_max = 2025
todos_os_anos = range(int(ano_min), int(ano_max) + 1)

# Cria um "catálogo" com todas as coordenadas/regiões únicas que existem no mapa
df_catalogo = df_consolidado[['lat', 'lon', 'country', 'state', 'county']].drop_duplicates()

In [ ]:
# Multiplica esse catálogo por todos os anos possíveis (Produto Cartesiano)
# Isso cria a "grade vazia" perfeita, onde cada país tem todos os anos
df_grade_temporal = df_catalogo.merge(pd.DataFrame({'ano': todos_os_anos}), how='cross')

# Junta a grade vazia perfeita com os nossos dados reais consolidados
print("Cruzando a grade temporal com as produtividades reais...")
df_global_yield = pd.merge(
    df_grade_temporal,
    df_consolidado,
    on=['lat', 'lon', 'country', 'state', 'county', 'ano'],
    how='left'
)

# Arredonda os valores que existem, e mantém os vazios estritamente como NaN (Não preencher com 0)
df_global_yield['yield (kg/ha)'] = df_global_yield['yield (kg/ha)'].round(0)

# 5. Organização Estrutural
print("Ordenando o painel de dados final...")
df_global_yield.sort_values(by=['country', 'state', 'county', 'ano'], inplace=True, ignore_index=True)

# ==============================================================================
# 6. DIAGNÓSTICO FINAL E SALVAMENTO
# ==============================================================================
print("\n================ RESUMO DO MASTER DATASET HARMONIZADO ===========")
print(f"Total de observações (Pixels x Todos os Anos): {len(df_global_yield)}")
print(f"Período Histórico Global: {int(ano_min)} até {int(ano_max)}")
print(f"Total de Registros com Produtividade (Validade): {df_global_yield['yield (kg/ha)'].count()}")
print(f"Total de Registros Inexistentes (NaN forçado): {df_global_yield['yield (kg/ha)'].isna().sum()}")
print("=================================================================\n")

# Para salvar no seu Drive
# caminho_salvamento = '/content/drive/Shareddrives/GAS-Henrique/MASTER_YIELD_GLOBAL_HARMONIZADO.csv'
# df_global_yield.to_csv(caminho_salvamento, index=False)
# print(f"✅ Base harmonizada salva com sucesso em: {caminho_salvamento}")

# Visualizar a prova do conceito (Ex: O México em 1940 estará com NaN)
df_global_yield

In [ ]:
df_global_yield.info()

In [ ]:
caminho_excel = os.path.join(os.getcwd(), 'inputs', 'data', 'coordinates.xlsx')

with pd.ExcelWriter(caminho_excel, mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
    df_global_yield.to_excel(writer, sheet_name='yield', index=False)

# Plotting map

In [ ]:
import os
import pandas as pd

caminho_excel = os.path.join(os.getcwd(),'inputs','data','coordinates.xlsx')
df = pd.read_excel(caminho_excel, sheet_name='yield')

In [ ]:
import plotly.express as px

# É recomendável filtrar por um ano específico (ex: 2021) para o mapa não sobrepor os anos
df_mapa = df[df['ano'] == 2000].copy()

# Cria o mapa interativo
fig = px.scatter_map(
    df_mapa,
    lat="lat",
    lon="lon",
    hover_name="county",
    hover_data=["state", "yield (kg/ha)"],
    color="yield (kg/ha)",
    size_max=30,
    zoom=1,
    center={"lat": 0, "lon": 0},
    map_style="carto-positron",
    title="Distribuição da Produtividade na Grade ERA5 (0.25°) - mundo"
)

fig.update_layout(margin={"r":0,"t":40,"l":0,"b":0})
fig.show()

# Segmentando os dados diariamente
NETCDF4 <br>
YIELD

In [19]:
import pandas as pd
import xarray as xr
import numpy as np
import os
import glob
import shutil
import tempfile
from tqdm.auto import tqdm

print("=== CONVERTENDO MASTER DATASET (ARQUITETURA DESACOPLADA) ===")

# 1. Carregar o arquivo
caminho_excel = os.path.join(os.getcwd(), 'inputs', 'data', 'coordinates.xlsx')
df = pd.read_excel(caminho_excel, sheet_name='yield')

df = df.dropna(subset=['ano'])
df['ano'] = df['ano'].astype(int)
ano_min, ano_max = df['ano'].min(), df['ano'].max()

if 'yield (kg/ha)' in df.columns:
    df.rename(columns={'yield (kg/ha)': 'yield'}, inplace=True)

# 2. Isolando metadados estáticos (O Mapa de Textos)
print("Isolando metadados geográficos...")
df_static = df[['lat', 'lon', 'country', 'state', 'county']].drop_duplicates(subset=['lat', 'lon'])
ds_static = df_static.set_index(['lat', 'lon']).to_xarray()

# Formato object suporta acentuação (Unicode) nativamente no h5netcdf
for var in ['country', 'state', 'county']:
    if var in ds_static:
        ds_static[var] = ds_static[var].astype(object)

base_lat = ds_static.lat
base_lon = ds_static.lon

# 3. Preparando Saída e Limpando Lixo Antigo
caminho_nc = os.path.join(os.getcwd(), 'inputs', 'data', 'yield', 'MASTER_YIELD_DIARIO.nc4')

# CORREÇÃO APLICADA AQUI: Tirando a pasta temporária do OneDrive
pasta_temporaria = os.path.join(tempfile.gettempdir(), 'temp_nc4_yield_doutorado')

# Deleta a pasta temporária antiga e cria uma nova limpa na raiz do Windows
shutil.rmtree(pasta_temporaria, ignore_errors=True)
os.makedirs(pasta_temporaria, exist_ok=True)

df_yield = df[['ano', 'lat', 'lon', 'yield']].drop_duplicates(subset=['ano', 'lat', 'lon'])

encoding_numerico = {'yield': {'zlib': True, 'complevel': 5, '_FillValue': np.nan, 'dtype': 'float32'}}

# ==============================================================================
# 4. CONSTRUÇÃO ANO A ANO (SOMENTE NÚMEROS, ZERO TEXTO)
# ==============================================================================
print("\nGerando blocos anuais de produtividade...")

for ano in tqdm(range(ano_min, ano_max + 1), desc="Processando Anos"):
    df_ano = df_yield[df_yield['ano'] == ano]

    if not df_ano.empty:
        ds_ano = df_ano.set_index(['lat', 'lon'])[['yield']].to_xarray()
        ds_ano = ds_ano.reindex(lat=base_lat, lon=base_lon)
    else:
        ds_ano = xr.Dataset({
            'yield': xr.DataArray(
                np.full((len(base_lat), len(base_lon)), np.nan, dtype=np.float32),
                coords=[base_lat, base_lon], dims=['lat', 'lon']
            )
        })

    ds_ano['yield'] = ds_ano['yield'].astype(np.float32)
    datas_ano = pd.date_range(start=f"{ano}-01-01", end=f"{ano}-12-31", freq='D')

    # Expande o tempo (Mas como não há textos, a RAM consumida é quase 0)
    ds_diario = ds_ano.expand_dims({'time': datas_ano})

    caminho_temp_ano = os.path.join(pasta_temporaria, f'yield_{ano}.nc4')
    ds_diario.to_netcdf(caminho_temp_ano, engine='h5netcdf', encoding=encoding_numerico)

    ds_ano.close()
    ds_diario.close()

# ==============================================================================
# 5. COSTURA LAZY E FUSÃO DOS TEXTOS
# ==============================================================================
print("\nCosturando anos e aplicando metadados (Dask Lazy Loading)...")

arquivos_anuais = sorted(glob.glob(os.path.join(pasta_temporaria, 'yield_*.nc4')))

# Abre todos os números levíssimos sem estourar a memória
ds_global = xr.open_mfdataset(arquivos_anuais, combine='by_coords', engine='h5netcdf')

# "Veste" o cubo com os textos dos países apenas nas coordenadas lat/lon
ds_final = xr.merge([ds_global, ds_static])
ds_final['yield'].attrs['units'] = 'kg/ha'
ds_final['yield'].attrs['description'] = 'Annual crop yield broadcasted to daily scale'

print("Salvando o Master Dataset final no disco (Isso levará alguns minutos)...")
# O Dask fará o trabalho duro silenciosamente enviando pedaço por pedaço ao disco
ds_final.to_netcdf(caminho_nc, engine='h5netcdf', encoding=encoding_numerico)

ds_global.close()
ds_final.close()

shutil.rmtree(pasta_temporaria, ignore_errors=True)

print(f"\n✅ Cubo NetCDF Diário gerado com sucesso!")
print(f"Salvo em: {caminho_nc}")

=== CONVERTENDO MASTER DATASET (ARQUITETURA DESACOPLADA) ===
Isolando metadados geográficos...

Gerando blocos anuais de produtividade...


Processando Anos:   0%|          | 0/86 [00:00<?, ?it/s]


Costurando anos e aplicando metadados (Dask Lazy Loading)...
Salvando o Master Dataset final no disco (Isso levará alguns minutos)...

✅ Cubo NetCDF Diário gerado com sucesso!
Salvo em: D:\OneDrive\Documentos\Git\Doutorado\inputs\data\yield\MASTER_YIELD_DIARIO.nc4


ERA5

In [1]:
import xarray as xr
import os
import glob
import pandas as pd
import numpy as np

print("=== INICIALIZANDO ASSOCIAÇÃO CLIMÁTICA (ERA5 LOCAL + YIELD) ===")

# ==========================================
# 1. DEFINIÇÃO DE CAMINHOS (PASTAS)
# ==========================================
# Caminho do seu arquivo mestre de produtividade
caminho_yield = os.path.join(os.getcwd(), 'inputs', 'data', 'yield', 'MASTER_YIELD_DIARIO.nc4')

# NOVO CAMINHO: Dados climáticos totalmente fora do OneDrive
pasta_era5 = r'D:\ERA5'

# ==========================================
# 2. CARREGAMENTO DO CUBO ALVO (Y)
# ==========================================
print("Carregando o Cubo de Produtividade Mestre...")
ds_yield = xr.open_dataset(caminho_yield, engine='h5netcdf')

# ==========================================
# 3. MAPEAMENTO VIRTUAL DO CLIMA (X)
# ==========================================
print("Mapeando virtualmente as variáveis do ERA5 (Lazy Loading)...")
VARIAVEIS_CLIMA = ['tasmax', 'tasmin', 'hurs', 'sfcWind', 'rsds', 'pr']
dict_clima = {}

for var in VARIAVEIS_CLIMA:
    # Busca os arquivos anuais dentro da pasta de cada variável
    padrao_busca = os.path.join(pasta_era5, var, f"ERA5_{var}_*.nc4")
    arquivos = sorted(glob.glob(padrao_busca))

    if not arquivos:
        print(f" ⚠️ Alerta: Nenhum arquivo encontrado para a variável: {var}")
        continue

    # 'chunks=auto' faz o Dask respeitar o tamanho nativo de cada ano (bissexto ou não)
    ds_var = xr.open_mfdataset(arquivos, engine='h5netcdf', chunks='auto')
    dict_clima[var] = ds_var[var]

# Unifica as 6 variáveis climáticas em um único bloco virtual
ds_clima_global = xr.Dataset(dict_clima)

# ==========================================
# 4. ALINHAMENTO GEOGRÁFICO EXPRESSO
# ==========================================
print("Alinhando a grade do ERA5 com o seu grid de estudo...")

# O reindex ajusta as latitudes e longitudes por vizinho mais próximo (nearest)
# Isso corrige micro-diferenças de arredondamento de float entre os datasets
ds_clima_alinhado = ds_clima_global.reindex(
    lat=ds_yield.lat,
    lon=ds_yield.lon,
    method='nearest'
)

# Fusão Virtual: Une a Produtividade (Y) e o Clima (X) no mesmo objeto
ds_dataset_ml = xr.merge([ds_yield, ds_clima_alinhado])

print("\n=== ESTRUTURA DO DATASET DE MACHINE LEARNING ===")
print(ds_dataset_ml)

# ==========================================
# 5. TESTE DE EXTRAÇÃO DE DADOS (PANDAS)
# ==========================================
print("\n[VALIDAÇÃO] Convertendo uma amostra diária para conferência...")

# Seleciona um dia específico na história para testar o cruzamento
data_amostra = '2003-06-15'
ds_dia = ds_dataset_ml.sel(time=data_amostra)

# Transforma o mapa bidimensional desse dia em uma tabela clássica
df_dia = ds_dia.to_dataframe().reset_index()

# Remove os pontos vazios (como oceanos ou regiões sem dados de yield)
df_treinamento_dia = df_dia.dropna(subset=['yield'])

print(f" -> No dia {data_amostra}, existem {len(df_treinamento_dia)} pixels agrícolas ativos.")
print("\n-> Amostra da tabela final que será enviada para a Inteligência Artificial:")
print(df_treinamento_dia[['lat', 'lon', 'country', 'state', 'county', 'yield', 'tasmax', 'pr']].head())

# Fechar conexões de leitura de arquivos por segurança
ds_yield.close()
ds_clima_global.close()

=== INICIALIZANDO ASSOCIAÇÃO CLIMÁTICA (ERA5 LOCAL + YIELD) ===
Carregando o Cubo de Produtividade Mestre...
Mapeando virtualmente as variáveis do ERA5 (Lazy Loading)...
Alinhando a grade do ERA5 com o seu grid de estudo...

=== ESTRUTURA DO DATASET DE MACHINE LEARNING ===
<xarray.Dataset> Size: 93GB
Dimensions:      (time: 31412, lat: 260, lon: 405)
Coordinates:
  * time         (time) datetime64[ns] 251kB 1940-01-01 ... 2025-12-31
  * lat          (lat) float64 2kB -40.88 -39.38 -38.88 ... 51.12 51.38 51.62
  * lon          (lon) float64 3kB -108.9 -108.6 -108.4 ... 134.1 134.4 134.6
    spatial_ref  int64 8B 0
Data variables:
    yield        (time, lat, lon) float32 13GB ...
    country      (lat, lon) <U13 5MB ...
    state        (lat, lon) <U19 8MB ...
    county       (lat, lon) <U32 13MB ...
    tasmax       (time, lat, lon) float32 13GB dask.array<chunksize=(108, 260, 405), meta=np.ndarray>
    tasmin       (time, lat, lon) float32 13GB dask.array<chunksize=(108, 260, 405), m

In [3]:
df_treinamento_dia.info()

<class 'pandas.DataFrame'>
Index: 3568 entries, 102 to 105106
Data columns (total 14 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   lat          3568 non-null   float64       
 1   lon          3568 non-null   float64       
 2   yield        3568 non-null   float32       
 3   country      3568 non-null   str           
 4   state        3568 non-null   str           
 5   county       3568 non-null   str           
 6   time         3568 non-null   datetime64[ns]
 7   spatial_ref  3568 non-null   int64         
 8   tasmax       3558 non-null   float32       
 9   tasmin       3558 non-null   float32       
 10  hurs         3558 non-null   float32       
 11  sfcWind      3558 non-null   float32       
 12  rsds         3558 non-null   float32       
 13  pr           3558 non-null   float32       
dtypes: datetime64[ns](1), float32(7), float64(2), int64(1), str(3)
memory usage: 320.6 KB


In [8]:
ds_dataset_ml.info()

xarray.Dataset {
dimensions:
	time = 31412 ;
	lat = 260 ;
	lon = 405 ;

variables:
	float32 yield(time, lat, lon) ;
		yield:units = kg/ha ;
		yield:description = Annual crop yield broadcasted to daily scale ;
	<U13 country(lat, lon) ;
	<U19 state(lat, lon) ;
	<U32 county(lat, lon) ;
	datetime64[ns] time(time) ;
	float64 lat(lat) ;
	float64 lon(lon) ;
	int64 spatial_ref() ;
		spatial_ref:crs_wkt = GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST],AUTHORITY["EPSG","4326"]] ;
		spatial_ref:semi_major_axis = 6378137.0 ;
		spatial_ref:semi_minor_axis = 6356752.314245179 ;
		spatial_ref:inverse_flattening = 298.257223563 ;
		spatial_ref:reference_ellipsoid_name = WGS 84 ;
		spatial_ref:longitude_of_prime_meridian = 0.0 ;
		spatial_ref:prime_meridian_name = Greenwich ;


In [11]:
import plotly.express as px
import numpy as np

# ==============================================================================
# 6. GERAÇÃO DO MAPA DE AUDITORIA INTERATIVO (ABRE NO NAVEGADOR VIA PYCHARM)
# ==============================================================================
# Vamos escolher um dia específico para auditar (ex: 15 de Junho de 2021)
data_mapa = '2023-01-01'
print(f"\n[MAPA] Extraindo dados globais para o dia {data_mapa}...")

# Isola o dia no supercubo virtual de 93GB
ds_dia_mapa = ds_dataset_ml.sel(time=data_mapa)

# Transforma apenas esse dia em DataFrame (pesa poucos Kilobytes na RAM)
df_mapa = ds_dia_mapa.to_dataframe().reset_index()

# O FILTRO MÁGICO: Remove oceanos e foca estritamente onde você tem dados agrícolas
df_mapa = df_mapa.dropna(subset=['yield'])

print(f"[MAPA] Renderizando {len(df_mapa)} pixels agrícolas no mapa interativo...")

# Criando o mapa geográfico interativo
fig = px.scatter_map(
    df_mapa,
    lat="lat",
    lon="lon",
    hover_name="county", # O título do balão será o Município
    hover_data={
        "country": True,   # Mostra o País
        "state": True,     # Mostra o Estado
        "yield": ":.0f",   # Mostra o Yield sem casas decimais (kg/ha)
        "tasmax": ":.2f",  # Mostra a Temp Máxima do dia (°C)
        "pr": ":.2f",      # Mostra a Chuva do dia (mm/day)
        "lat": False,      # Oculta das linhas extras pois já está no topo
        "lon": False
    },
    color="yield",         # Os pontos serão coloridos pela produtividade
    color_continuous_scale=px.colors.sequential.YlGn, # Escala de verde (agrícola)
    zoom=1.5,
    center={"lat": 20, "lon": 0}, # Centro do mapa-múndi
    title=f"Auditoria do Master Dataset: Produtividade (Anual) e Clima no dia {data_mapa}"
)

# Ajusta o tamanho dos pontos na tela para ficarem visíveis como uma grade (grid)
fig.update_traces(marker=dict(size=6, opacity=0.8))
fig.update_layout(margin={"r":0, "t":40, "l":0, "b":0})

print("✅ Mapa gerado! Uma aba será aberta no seu navegador padrão para inspeção visual...")
# No PyCharm, este comando vai disparar a abertura do navegador automaticamente
fig.show()


[MAPA] Extraindo dados globais para o dia 2023-01-01...
[MAPA] Renderizando 1754 pixels agrícolas no mapa interativo...
✅ Mapa gerado! Uma aba será aberta no seu navegador padrão para inspeção visual...


In [12]:
print("\n=== SALVANDO O MASTER DATASET BRUTO (ÚNICO ARQUIVO) ===")

caminho_master_nc = os.path.join(os.getcwd(), 'inputs', 'data', 'yield', 'dados_brutos.nc4')

# Configuração de compressão para manter o tamanho razoável
encoding_bruto = {
    'yield': {'zlib': True, 'complevel': 5, '_FillValue': np.nan, 'dtype': 'float32'},
    'tasmax': {'zlib': True, 'complevel': 5, 'dtype': 'float32'},
    'tasmin': {'zlib': True, 'complevel': 5, 'dtype': 'float32'},
    'hurs': {'zlib': True, 'complevel': 5, 'dtype': 'float32'},
    'pr': {'zlib': True, 'complevel': 5, 'dtype': 'float32'},
    'sfcWind': {'zlib': True, 'complevel': 5, 'dtype': 'float32'},
    'rsds': {'zlib': True, 'complevel': 5, 'dtype': 'float32'}
}

print("A gravar 28 GB no disco (Isto vai demorar algum tempo, deixe o PyCharm processar)...")

# O Dask vai calcular a fusão e gravar diretamente no NetCDF4
ds_dataset_ml.to_netcdf(caminho_master_nc, engine='h5netcdf', encoding=encoding_bruto)

print(f"✅ Arquivo único bruto salvo com sucesso em: {caminho_master_nc}")


=== SALVANDO O MASTER DATASET BRUTO (ÚNICO ARQUIVO) ===
A gravar 93 GB no disco (Isto vai demorar algum tempo, deixe o PyCharm processar)...
✅ Arquivo único bruto salvo com sucesso em: D:\OneDrive\Documentos\Git\Doutorado\inputs\data\yield\dados_brutos.nc4
